In [8]:
import os
import getpass
import subprocess
import sys

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    print("python-dotenv is not installed. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file. Please create a .env file with your API key.")

os.environ["GROQ_API_KEY"] = api_key

assert os.getenv("GROQ_API_KEY"), "Groq API key is missing."
print("Groq API key is configured from .env file.")

Groq API key is configured from .env file.


In [9]:
import os
import time
from pathlib import Path
from typing import TypedDict

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

GROQ_MODEL = "openai/gpt-oss-120b"

MODEL_CANDIDATES = [
    "openai/gpt-oss-20b",
    "llama-3.1-8b-instant",
    "canopylabs/orpheus-arabic-saudi",
]

MODEL_COST_RATES = {
    "openai/gpt-oss-20b": {"input": 0.00015, "output": 0.0006},
    "llama-3.1-8b-instant": {"input": 0.00005, "output": 0.00008},
    "canopylabs/orpheus-arabic-saudi": {"input": 0.00009, "output": 0.00018},
}


class QAAgentState(TypedDict):
    requirement: str
    analysis: str
    test_cases: str
    security_review: str
    review: str


def create_model(model_name: str):
    return ChatGroq(
        model=model_name,
        temperature=0.2,
        max_tokens=1800,
        reasoning_format="parsed",
        max_retries=2,
    )


def call_specialist(system_prompt, task, model_name=None):
    model_to_use = model_name or GROQ_MODEL
    candidates = [model_to_use] + [m for m in MODEL_CANDIDATES if m != model_to_use]
    last_error = None
    for candidate in candidates:
        try:
            response = create_model(candidate).invoke([
                ("system", system_prompt),
                ("human", task),
            ])
            return response.content
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"All models failed. Last error: {last_error}")


def estimate_tokens(text: str) -> int:
    return max(1, len(text.split()) + len(text) // 4)


def estimate_cost(model_name: str, prompt_text: str, response_text: str) -> float:
    rates = MODEL_COST_RATES[model_name]
    input_tokens = estimate_tokens(prompt_text)
    output_tokens = estimate_tokens(response_text)
    return round((input_tokens * rates["input"] + output_tokens * rates["output"]) / 1000, 6)


def score_quality(text: str, expected_keywords: list[str]) -> float:
    lowered = text.lower()
    matched = sum(1 for keyword in expected_keywords if keyword.lower() in lowered)
    return round(min(100.0, 60.0 + (matched / max(1, len(expected_keywords))) * 40.0), 2)


def score_latency(latency_ms: float) -> float:
    if latency_ms is None:
        return 0.0
    return round(max(0.0, 100.0 - min(latency_ms / 10.0, 100.0)), 2)


def score_cost(cost_usd: float) -> float:
    return round(max(0.0, 100.0 - min(cost_usd * 100000.0, 100.0)), 2)


def evaluate_model_for_agent(agent_name: str, system_prompt: str, task: str, model_name: str):
    start = time.perf_counter()
    try:
        response_text = call_specialist(system_prompt, task, model_name=model_name)
        latency_ms = round((time.perf_counter() - start) * 1000, 1)
    except Exception as exc:
        return {
            "model": model_name,
            "success": False,
            "error": str(exc),
            "quality_score": 0.0,
            "latency_score": 0.0,
            "cost_score": 0.0,
            "composite_score": 0.0,
        }

    quality_keywords = {
        "requirements_analyst": ["actors", "business rules", "acceptance criteria", "risks", "dependencies", "ambiguities"],
        "test_designer": ["scenario", "preconditions", "expected result", "priority"],
        "security_reviewer": ["security", "privacy", "access control", "session", "data exposure"],
        "qa_reviewer": ["approve", "revise", "coverage", "edge cases", "testability"],
    }[agent_name]

    quality_score = score_quality(response_text, quality_keywords)
    cost = estimate_cost(model_name, system_prompt + "\n\n" + task, response_text)
    cost_score = score_cost(cost)
    latency_score = score_latency(latency_ms)
    weights = {
        "requirements_analyst": (0.55, 0.25, 0.20),
        "test_designer": (0.45, 0.30, 0.25),
        "security_reviewer": (0.60, 0.20, 0.20),
        "qa_reviewer": (0.50, 0.30, 0.20),
    }[agent_name]
    composite_score = round(
        quality_score * weights[0] + latency_score * weights[1] + cost_score * weights[2],
        2,
    )
    return {
        "model": model_name,
        "success": True,
        "latency_ms": latency_ms,
        "quality_score": quality_score,
        "cost_usd": cost,
        "latency_score": latency_score,
        "cost_score": cost_score,
        "composite_score": composite_score,
        "response": response_text,
    }


def select_best_model(agent_name: str, system_prompt: str, task: str):
    results = [evaluate_model_for_agent(agent_name, system_prompt, task, model) for model in MODEL_CANDIDATES]
    successful_results = [item for item in results if item["success"]]
    if not successful_results:
        raise RuntimeError("All models failed for the requested agent.")
    return sorted(successful_results, key=lambda item: item["composite_score"], reverse=True)[0]


def make_requirements_analyst(selected_model: str):
    def requirements_analyst(state: QAAgentState):
        analysis = call_specialist(
            "You are a senior QA requirements analyst. Identify actors, business rules, acceptance criteria, risks, dependencies, and ambiguous requirements. Be concise and do not invent missing facts.",
            f"Analyze this requirement for testing:\n\n{state['requirement']}",
            model_name=selected_model,
        )
        return {"analysis": analysis}

    return requirements_analyst


def make_test_designer(selected_model: str):
    def test_designer(state: QAAgentState):
        test_cases = call_specialist(
            "You are a senior test designer. Produce a compact Markdown table with ID, scenario, preconditions, steps, expected result, test type, and priority. Cover positive, negative, boundary, security, and failure paths.",
            f"Requirement:\n{state['requirement']}\n\nRequirements analysis:\n{state['analysis']}\n\nDesign executable test cases.",
            model_name=selected_model,
        )
        return {"test_cases": test_cases}

    return test_designer


def make_security_reviewer(selected_model: str):
    def security_reviewer(state: QAAgentState):
        security_review = call_specialist(
            "You are a security reviewer for software requirements. Identify security, privacy, access control, session handling, and data exposure risks. Highlight missing safeguards and note any conflicting or vague requirements.",
            f"Requirement:\n{state['requirement']}\n\nRequirements analysis:\n{state['analysis']}\n\nProposed tests:\n{state['test_cases']}",
            model_name=selected_model,
        )
        return {"security_review": security_review}

    return security_reviewer


def make_qa_reviewer(selected_model: str):
    def qa_reviewer(state: QAAgentState):
        review = call_specialist(
            "You are a critical QA lead. Review the proposed tests for requirement coverage, missing edge cases, duplication, testability, and business risk. Finish with APPROVE or REVISE and a short reason.",
            f"Requirement:\n{state['requirement']}\n\nAnalysis:\n{state['analysis']}\n\nProposed tests:\n{state['test_cases']}\n\nSecurity review:\n{state['security_review']}",
            model_name=selected_model,
        )
        return {"review": review}

    return qa_reviewer


print("Extended QA chain is ready with model comparison and four agents.")


Extended QA chain is ready with model comparison and four agents.


In [11]:
from pathlib import Path

requirements_path = Path("requirements_document.md")
if not requirements_path.exists():
    requirements_path = Path("requirements_document.txt")

if not requirements_path.exists():
    raise FileNotFoundError("Could not find a requirements document. Create requirements_document.md or requirements_document.txt in the workspace.")

requirements_text = requirements_path.read_text(encoding="utf-8")

agent_prompts = {
    "requirements_analyst": "You are a senior QA requirements analyst. Identify actors, business rules, acceptance criteria, risks, dependencies, and ambiguous requirements. Be concise and do not invent missing facts.",
    "test_designer": "You are a senior test designer. Produce a compact Markdown table with ID, scenario, preconditions, steps, expected result, test type, and priority. Cover positive, negative, boundary, security, and failure paths.",
    "security_reviewer": "You are a security reviewer for software requirements. Identify security, privacy, access control, session handling, and data exposure risks. Highlight missing safeguards and note conflicting or vague requirements.",
    "qa_reviewer": "You are a critical QA lead. Review the proposed tests for requirement coverage, missing edge cases, duplication, testability, and business risk. Finish with APPROVE or REVISE and a short reason.",
}

agent_tasks = {
    "requirements_analyst": f"Analyze this requirement for testing:\n\n{requirements_text}",
    "test_designer": f"Requirement:\n{requirements_text}\n\nDesign executable test cases.",
    "security_reviewer": f"Requirement:\n{requirements_text}\n\nReview the requirement from a security perspective.",
    "qa_reviewer": f"Requirement:\n{requirements_text}\n\nReview the overall QA plan.",
}

selected_models = {}
for agent_name in ["requirements_analyst", "test_designer", "security_reviewer", "qa_reviewer"]:
    best = select_best_model(agent_name, agent_prompts[agent_name], agent_tasks[agent_name])
    selected_models[agent_name] = best
    print(f"{agent_name}: {best['model']} | composite={best['composite_score']} | quality={best['quality_score']} | latency={best['latency_ms']}ms | cost=${best['cost_usd']}")

builder = StateGraph(QAAgentState)
builder.add_node("requirements_analyst", make_requirements_analyst(selected_models["requirements_analyst"]["model"]))
builder.add_node("test_designer", make_test_designer(selected_models["test_designer"]["model"]))
builder.add_node("security_reviewer", make_security_reviewer(selected_models["security_reviewer"]["model"]))
builder.add_node("qa_reviewer", make_qa_reviewer(selected_models["qa_reviewer"]["model"]))
builder.add_edge(START, "requirements_analyst")
builder.add_edge("requirements_analyst", "test_designer")
builder.add_edge("test_designer", "security_reviewer")
builder.add_edge("security_reviewer", "qa_reviewer")
builder.add_edge("qa_reviewer", END)

qa_agent_chain = builder.compile()

result = qa_agent_chain.invoke({
    "requirement": requirements_text,
    "analysis": "",
    "test_cases": "",
    "security_review": "",
    "review": "",
})

output_file = Path("qa_agent_output.txt")
with open(output_file, "w", encoding="utf-8") as f:
    for heading, key in [
        ("REQUIREMENTS ANALYST", "analysis"),
        ("TEST DESIGNER", "test_cases"),
        ("SECURITY REVIEWER", "security_review"),
        ("QA REVIEWER", "review"),
    ]:
        separator = f"\n{'=' * 20} {heading} {'=' * 20}\n"
        f.write(separator)
        f.write(result[key])
        f.write("\n")

        print(separator)
        print(result[key])

print(f"\n✓ Output also saved to qa_agent_output.txt")
print("Selected models:")
for agent_name, metric in selected_models.items():
    print(f"- {agent_name}: {metric['model']} (composite={metric['composite_score']})")


requirements_analyst: llama-3.1-8b-instant | composite=71.74 | quality=100.0 | latency=2739.0ms | cost=$0.000163
test_designer: llama-3.1-8b-instant | composite=65.67 | quality=100.0 | latency=4971.8ms | cost=$0.000173
security_reviewer: llama-3.1-8b-instant | composite=60.42 | quality=68.0 | latency=14744.4ms | cost=$1.9e-05
qa_reviewer: canopylabs/orpheus-arabic-saudi | composite=68.84 | quality=100.0 | latency=4392.4ms | cost=$5.8e-05

==================== REQUIREMENTS ANALYST ====================

**Actors**  
| Actor | Role | Notes |
|-------|------|-------|
| Registered Customer | Initiator | Requests password reset |
| System (Password‑Reset Service) | Processor | Generates token, sends email, validates link |
| Email Service | Delivery | Sends reset email |
| Business Team | Decision‑maker | Provides policy on account lock and password rules |

---

### Business Rules (as stated)

| # | Rule | Source |
|---|------|--------|
| 1 | Reset link expires **15 min** after issuance. | 